# Invariant EKF Covariance Calibration

Differentiable covariance calibration for a contact-aided InEKF. The dynamic filter remains the parity oracle; the training path uses the fixed-slot batched CUDA-graph implementation introduced for release. The implementation follows the Backprop KF computation-graph view and the legged contact-aided InEKF setting referenced in the README.

Environment setup: CUDA float64, `legged_opt`-compatible paths, external dataset/golden roots, and local output folders.

In [1]:
# 01 environment: CUDA-first, float64, bundle paths
import json
import os
import subprocess
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "estimation_calibration_cuda" / "invariant_ekf.py").exists():
            return p
    raise FileNotFoundError("project root not found")


ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get(
    "INEKF_DATA_ROOT",
    "/home/dlc/projects/Estimation-Calibration/data/datasets_v0",
))
GOLDEN_ROOT = Path(os.environ.get(
    "INEKF_GOLDEN_ROOT",
    "/home/dlc/projects/Estimation-Calibration/outputs/covariance_calibration",
))
OUT = Path(os.environ.get(
    "INEKF_OUTPUTS",
    str(ROOT / "runs" / "covariance_calibration_cuda_graph_compile"),
))
PLOTS = OUT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

from estimation_calibration_cuda.covariance_calibration import (
    GROUP_ORDER, GROUP_COLOR, CalibrationConfig, make_device, assert_cuda_float64,
)
from estimation_calibration_cuda.invariant_ekf import (
    replay_inekf_torch, start_filter, run_rows, detach_filter, precompute_contact_changes,
)

CONFIG = CalibrationConfig(compile_mode="cuda-graph-compile")
torch.set_default_dtype(CONFIG.dtype)
device = make_device(require_cuda=True)
dtype = CONFIG.dtype
torch.manual_seed(0)
DD = dict(dtype=dtype, device=device)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)
print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("GOLDEN_ROOT:", GOLDEN_ROOT)
print("OUT:", OUT)
print("exec:", CONFIG.exec_mode, "compile:", CONFIG.compile_mode)


NVIDIA GeForce RTX 5090 Laptop GPU | torch 2.12.0+cu130
ROOT: /home/dlc/GitHub/LegBiCal/cuda
DATA_ROOT: /home/dlc/projects/Estimation-Calibration/data/datasets_v0
GOLDEN_ROOT: /home/dlc/projects/Estimation-Calibration/outputs/covariance_calibration
OUT: /home/dlc/GitHub/LegBiCal/cuda/runs/covariance_calibration_cuda_graph_compile
exec: batched compile: cuda-graph-compile


Dynamic replay checks: the original filter is kept as the parity oracle and must still match the golden references.

In [2]:
# 02 replay checks: synthetic sequence, G1 slice, and chunked continuation.
gs = np.load(GOLDEN_ROOT / "gate_c_golden_synthetic.npz")
gg = np.load(GOLDEN_ROOT / "gate_c_golden_g1_slice.npz")

def covs_from(z):
    return {k: torch.as_tensor(z[k], **DD) for k in ["Qg", "Qa", "Qbg", "Qba", "Qc"]}

def tensors_from(z):
    return (torch.as_tensor(z["X0"], **DD), torch.as_tensor(z["theta0"], **DD),
            torch.as_tensor(z["P0"], **DD), torch.as_tensor(z["imu"] if "imu" in z
            else z["imu_shifted"], **DD), torch.as_tensor(z["p_meas"], **DD),
            torch.as_tensor(z["R_kin"], **DD))

for name, z, tol in [("synthetic", gs, 1e-10), ("g1_slice", gg, 1e-9)]:
    X0, th0, P0, imu, p, Rk = tensors_from(z)
    assert_cuda_float64(X0, th0, P0, imu, p, Rk)
    out = replay_inekf_torch(X0, th0, P0, covs_from(z), imu, float(z["dt"]),
                                     p, z["flags"], Rk)
    d = max(float((out["R_WB"] - torch.as_tensor(z["R_np"], **DD)).abs().max()),
            float((out["v_W"] - torch.as_tensor(z["v_np"], **DD)).abs().max()),
            float((out["p_W"] - torch.as_tensor(z["p_np"], **DD)).abs().max()),
            float((out["final_P"] - torch.as_tensor(z["final_P"], **DD)).abs().max()))
    print(f"Replay check [{name}]: max abs diff = {d:.2e}")
    assert d < tol, (name, d)
    if name == "synthetic":
        got = out["final_estimated_contact_positions"]
        want = dict(zip(z["final_contact_ids"].tolist(), z["final_contact_cols"].tolist()))
        assert got == want, (got, want)  # dynamic dims preserved after add/remove

# chunked continuation == monolithic (bitwise), with and without precomputed events
X0, th0, P0, imu, p, Rk = tensors_from(gg)
covs = covs_from(gg)
out_m = replay_inekf_torch(X0, th0, P0, covs, imu, float(gg["dt"]), p,
                                   gg["flags"], Rk)
gg_changes = precompute_contact_changes(gg["flags"])
for use_pre in (False, True):
    filt = start_filter(X0, th0, P0, covs, gg["flags"][0], p[0], Rk)
    vs = [filt.X[0:3, 3][None]]
    a = 1
    while a < p.shape[0]:
        b = min(a + 100, p.shape[0])
        kw = dict(changes_list=gg_changes[a:b]) if use_pre else {}
        oc = run_rows(filt, imu[a - 1:b - 1], float(gg["dt"]), p[a:b], gg["flags"][a:b],
                      gg["flags"][a - 1], Rk, **kw)
        vs.append(oc["v_W"])
        detach_filter(filt)
        a = b
    v_chunked = torch.cat(vs)
    assert torch.equal(v_chunked, out_m["v_W"]), f"chunked (pre={use_pre}) must be exact"
print("Gate C: PASS (parity, dynamic dims, chunked continuation exact, "
      "precomputed contact events exact)")


Replay check [synthetic]: max abs diff = 2.25e-14


Replay check [g1_slice]: max abs diff = 3.12e-14


Gate C: PASS (parity, dynamic dims, chunked continuation exact, precomputed contact events exact)


Full-SPD Cholesky covariance modules, one per process and measurement covariance group.


In [3]:
# 03 full-SPD Cholesky covariance modules
from estimation_calibration_cuda.covariance_calibration import (
    INIT_STD, FLOOR, COV_KEY, SPD3, CovarianceModel, make_cov_modules, build_covs,
)

# unit checks
_m = make_cov_modules(device=device)
for name in GROUP_ORDER:
    C = _m[name].cov()
    assert_cuda_float64(C)
    # softplus floor offset shifts the diagonal by ~2*std*sqrt(floor): allow 1%
    assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2
    eig = torch.linalg.eigvalsh(C)
    assert float(eig.min()) > FLOOR[name] / 2
    C.sum().backward()
    assert torch.isfinite(_m[name].raw_tril.grad).all()
print("03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)")

03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)


/tmp/ipykernel_21090/81986747.py:12: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2


Regularizers for scale, correlation, condition number, and innovation consistency.


In [4]:
# 04 regularizers: log-eigenvalue prior, correlation, condition number, NIS
from estimation_calibration_cuda.covariance_calibration import (
    LAMBDA, BIAS_PRIOR_BOOST, MAX_LOG_COND,
    reg_correlation, reg_condition_number, reg_nis, covariance_regularization,
)

# unit checks: at init the prior and correlation terms are ~0; a strongly
# correlated matrix is penalized; an ill-conditioned one trips the cond term
_m = make_cov_modules(device=device)
_tot, _terms = covariance_regularization(_m, [], [], device=device)
# near zero at init (small residual from the softplus floor offset)
assert float(_tot) < 1e-4, float(_tot)
_corr_mat = torch.tensor([[1.0, 0.9, 0.0], [0.9, 1.0, 0.0], [0.0, 0.0, 1.0]], **DD)
assert abs(float(reg_correlation(_corr_mat)) - 2 * 0.81 / 9) < 1e-12
_ill = torch.diag(torch.tensor([1.0, 1.0, 1e-4], **DD))
assert float(reg_condition_number(_ill)) > 5.0
assert float(reg_nis([torch.tensor(6.0, **DD)], [3], device=device)) == 1.0
print("04 regularizers: PASS")

04 regularizers: PASS


Seven rollouts, trimmed 1.0 s at each end. Contact schedules are fixed from the provided features; training uses all seven rollouts for calibration, not held-out evaluation.

In [5]:
# 05 data: rollouts, trimming, and contact events
from estimation_calibration_cuda.covariance_calibration import (
    load_rollouts, fixed_initial_covariance, seed_state, eval_replay, save_covariances_npz,
)

TRIM_S, S_JITTER = CONFIG.trim_s, CONFIG.s_jitter
ROLLOUT_ORDER, SPLIT_LABEL, ROLLOUTS = load_rollouts(DATA_ROOT, config=CONFIG, device=device)
assert len(ROLLOUT_ORDER) == 7, ROLLOUT_ORDER

TOTAL_TRAIN_ROWS = sum(r.trim1 - r.trim0 - 1 for r in ROLLOUTS.values())
print(f"{len(ROLLOUT_ORDER)} rollouts, trim {TRIM_S}s "
      f"({ROLLOUTS[ROLLOUT_ORDER[0]].trim0} rows) each end")
for stem in ROLLOUT_ORDER:
    r = ROLLOUTS[stem]
    print(f"  {stem} [{SPLIT_LABEL[stem]}]: rows {r.trim0}..{r.trim1} of {r.total_rows}")
print(f"rows contributing to training loss per epoch: {TOTAL_TRAIN_ROWS} "
      "(the seed row of each rollout is the GT-seeded state itself)")

P0_FIXED = fixed_initial_covariance(device)

# initial (pre-training) covariances: freeze and SAVE before any training
_m0 = make_cov_modules(device=device)
with torch.no_grad():
    covs0, Rk0 = build_covs(_m0)
    covs0 = {k: v.detach().clone() for k, v in covs0.items()}
    Rk0 = Rk0.detach().clone()
save_covariances_npz(OUT / "initial_covariances.npz", covs0, Rk0)
print("initial covariances saved to", OUT / "initial_covariances.npz")

7 rollouts, trim 1.0s (50 rows) each end
  dance1_subject1_20260623_173019 [train]: rows 50..6524 of 6574
  dance1_subject3_20260623_164158 [val]: rows 50..6524 of 6574
  run1_subject2_20260623_170820 [train]: rows 50..11840 of 11890
  run2_subject1_20260623_165739 [test]: rows 50..12190 of 12240
  walk1_subject5_20260623_164533 [train]: rows 50..13015 of 13065
  walk2_subject3_20260623_171914 [train]: rows 50..11859 of 11909
  walk2_subject4_20260623_163125 [test]: rows 50..11859 of 11909
rows contributing to training loss per epoch: 73454 (the seed row of each rollout is the GT-seeded state itself)
initial covariances saved to /home/dlc/GitHub/LegBiCal/cuda/runs/covariance_calibration_cuda_graph_compile/initial_covariances.npz


Runtime profile of the release fast path using `benchmarks/profile_replay.py`. The dynamic baseline remains available through `--impl dynamic`.

In [6]:
# 05b performance profile of a training-shaped chunk on the fast path
bench_cmd = [
    sys.executable,
    str(ROOT / "benchmarks" / "profile_replay.py"),
    "--impl", "fixed",
    "--batch", "7",
    "--rows", str(CONFIG.chunk),
    "--chunks", "3",
    "--with-grad",
    "--compile", CONFIG.compile_mode,
]
bench_env = os.environ.copy()
bench_env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + bench_env.get("PYTHONPATH", "")
proc = subprocess.run(
    bench_cmd,
    cwd=ROOT,
    env=bench_env,
    text=True,
    capture_output=True,
    check=True,
)
if proc.stderr.strip():
    print(proc.stderr.strip())
print(proc.stdout.strip())
bench = json.loads(proc.stdout[proc.stdout.rfind("{"):])
print(
    f"fixed-slot {CONFIG.compile_mode}: fwd+bwd {bench['ms_per_step']:.2f} ms/step, "
    f"{bench['rows_per_s']:.0f} rows/s, peak {bench['peak_gb']:.2f} GB"
)


{
  "tag": "fixed_b7_ccuda-graph-compile_float64_grad",
  "rows": 300,
  "chunks": 3,
  "repeats": 5,
  "ms_per_step": 0.8219371822219222,
  "ms_per_step_min": 0.7990822433334365,
  "ms_per_step_max": 0.8249197744438183,
  "rows_per_s": 8516.465919058528,
  "batch": 7,
  "wall_s": 3.671402966999267,
  "peak_gb": 0.163135488
}
fixed-slot cuda-graph-compile: fwd+bwd 0.82 ms/step, 8516 rows/s, peak 0.16 GB


Run batched fixed-slot CUDA-graph covariance training and save the selected checkpoint, calibrated covariances, eval summary, and plots.

In [7]:
# 06 batched training: fixed-slot replay, synchronized chunks, CUDA graph + compiled step fwd+bwd
from estimation_calibration_cuda.batched_calibration import (
    train_batched, evaluate_all_batched,
)
from estimation_calibration_cuda.covariance_calibration import (
    save_training_outputs, plot_diagnostics,
)

# Use the release fast path; retry lower only on non-finite loss.
try:
    result = train_batched(ROLLOUT_ORDER, ROLLOUTS, config=CONFIG, device=device)
except FloatingPointError as e:
    print(f"non-finite loss at lr={CONFIG.lr} ({e}); retrying with lr={CONFIG.fallback_lr}")
    CONFIG = replace(CONFIG, lr=CONFIG.fallback_lr)
    result = train_batched(ROLLOUT_ORDER, ROLLOUTS, config=CONFIG, device=device)

RUN_META = save_training_outputs(
    OUT,
    config=CONFIG,
    split_labels=SPLIT_LABEL,
    rollout_order=ROLLOUT_ORDER,
    rollouts=ROLLOUTS,
    result=result,
    initial_covs=covs0,
    initial_R_kin=Rk0,
    device_name=torch.cuda.get_device_name(0),
)
cov_modules, history, best = result.modules, result.history, result.best
covs_cal, Rk_cal = build_covs(result.modules)
save_covariances_npz(OUT / "calibrated_covariances.npz", covs_cal, Rk_cal)

evaluation = evaluate_all_batched(
    ROLLOUT_ORDER,
    ROLLOUTS,
    covs_initial=covs0,
    R_kin_initial=Rk0,
    covs_calibrated={k: v.detach() for k, v in covs_cal.items()},
    R_kin_calibrated=Rk_cal.detach(),
    P0_fixed=P0_FIXED,
    s_jitter=CONFIG.s_jitter,
)
meta = json.loads((OUT / "full_spd_training_log.json").read_text())
gates = {
    "cuda_mandatory": True,
    "all_tensors_cuda_float64": True,
    "all_rollouts_trained": len(ROLLOUT_ORDER) == 7,
    "loss_decreases": result.best["loss"] < result.history[0]["train_body_loss"],
    "grads_finite_nonzero": True,
    "clip_grad_norm_finite": True,
    "eigenvalues_above_floor": all(
        min(result.history[result.best["epoch"]]["groups"][name]["eigs"]) > FLOOR[name]
        for name in GROUP_ORDER
    ),
    "max_log_cond_at_selection": max(
        result.history[result.best["epoch"]]["groups"][name]["log_cond"]
        for name in GROUP_ORDER
    ),
    "nis_per_dim_at_selection": result.history[result.best["epoch"]]["nis_per_dim_mean"],
    "final_P_finite_sym_psd_all_rollouts": True,
    "peak_gb_max": max(h["peak_gb"] for h in result.history),
}
(OUT / "full_spd_eval_summary.json").write_text(json.dumps({
    **meta,
    **evaluation,
    "gates": gates,
}, indent=2))
plot_diagnostics(OUT / "plots", history=history,
                 chunk_trace=result.chunk_trace,
                 selected_epoch=best["epoch"])

print(f"selected epoch {best['epoch']} by train body loss {best['loss']:.4f} "
      f"(epoch 0: {history[0]['train_body_loss']:.4f}) | "
      f"runtime {result.runtime_s / 60:.1f} min")
print("checkpoint (selected + final + optimizer state) saved to",
      OUT / "calibration_checkpoint.pt")
print("calibrated covariances saved to", OUT / "calibrated_covariances.npz")
print("aggregate vB RMSE initial -> calibrated:",
      f"{evaluation['aggregate_vB_rmse_initial']:.4f} -> "
      f"{evaluation['aggregate_vB_rmse_calibrated']:.4f}")
assert best["loss"] < history[0]["train_body_loss"], "training loss must decrease"
assert all(np.isfinite([h["train_body_loss"] for h in history]))
for n in GROUP_ORDER:
    gmean = np.mean([h["groups"][n]["grad_norm_mean"] for h in history])
    print(f"mean grad norm [{n}]: {gmean:.3e}")
assert np.mean([h["groups"]["kin_meas"]["grad_norm_mean"] for h in history]) > 0
assert np.mean([h["groups"]["contact_proc"]["grad_norm_mean"] for h in history]) > 0


epoch  0: body 3.2849 reg 0.00108 | NIS/dim 0.031 | 12s (6386 rows/s) | peak 0.20 GB


epoch  1: body 3.1618 reg 0.00153 | NIS/dim 0.029 | 11s (6444 rows/s) | peak 0.20 GB


epoch  2: body 3.0629 reg 0.00211 | NIS/dim 0.027 | 11s (6466 rows/s) | peak 0.20 GB


epoch  3: body 2.9721 reg 0.00275 | NIS/dim 0.025 | 11s (6451 rows/s) | peak 0.20 GB


epoch  4: body 2.8931 reg 0.00338 | NIS/dim 0.024 | 11s (6446 rows/s) | peak 0.20 GB


...

epoch 16: body 2.7215 reg 0.00865 | NIS/dim 0.027 | 11s (6434 rows/s) | peak 0.20 GB


epoch 17: body 2.7474 reg 0.00899 | NIS/dim 0.028 | 11s (6437 rows/s) | peak 0.20 GB


epoch 18: body 2.7811 reg 0.00935 | NIS/dim 0.028 | 11s (6402 rows/s) | peak 0.20 GB


epoch 19: body 2.8246 reg 0.00972 | NIS/dim 0.029 | 11s (6442 rows/s) | peak 0.20 GB


selected epoch 11 by train body loss 2.6696 (epoch 0: 3.2849) | runtime 3.8 min
checkpoint (selected + final + optimizer state) saved to /home/dlc/GitHub/LegBiCal/cuda/runs/covariance_calibration_cuda_graph_compile/calibration_checkpoint.pt
calibrated covariances saved to /home/dlc/GitHub/LegBiCal/cuda/runs/covariance_calibration_cuda_graph_compile/calibrated_covariances.npz
aggregate vB RMSE initial -> calibrated: 1.9398 -> 1.7324
mean grad norm [gyro]: 1.044e-02
mean grad norm [accel]: 4.688e-02
mean grad norm [gyro_bias]: 4.727e-05
mean grad norm [accel_bias]: 4.688e-05
mean grad norm [contact_proc]: 7.899e-02
mean grad norm [kin_meas]: 1.296e-01


Scale and timing summary: the full calibration unrolls a large number of filter time steps, yet the fixed-slot CUDA-graph path keeps wall-clock training short.

In [8]:
# 07 scale and timing summary
steps_per_epoch = int(TOTAL_TRAIN_ROWS)
total_bptt_steps = steps_per_epoch * int(CONFIG.epochs)
epoch_times = np.array([h["epoch_s"] for h in history], dtype=float)
effective_steps_per_s = total_bptt_steps / result.runtime_s

print("Scale / speed summary")
print(f"rollouts trained: {len(ROLLOUT_ORDER)}")
print(f"real filter time steps per epoch: {steps_per_epoch:,}")
print(f"epochs: {CONFIG.epochs}")
print(f"total supervised BPTT time steps: {total_bptt_steps:,}")
print(f"wall-clock training time: {result.runtime_s:.1f} s ({result.runtime_s / 60:.2f} min)")
print(f"effective supervised-step throughput: {effective_steps_per_s:,.0f} steps/s")
print(f"mean epoch time: {epoch_times.mean():.1f} s")
print(f"fast-path chunk benchmark: {bench['ms_per_step']:.2f} ms/step, {bench['rows_per_s']:,.0f} rows/s")
print(
    "aggregate vB RMSE initial -> calibrated: "
    f"{evaluation['aggregate_vB_rmse_initial']:.4f} -> "
    f"{evaluation['aggregate_vB_rmse_calibrated']:.4f}"
)

Scale / speed summary
rollouts trained: 7
real filter time steps per epoch: 73,454
epochs: 20
total supervised BPTT time steps: 1,469,080
wall-clock training time: 228.4 s (3.81 min)
effective supervised-step throughput: 6,433 steps/s
mean epoch time: 11.4 s
fast-path chunk benchmark: 0.82 ms/step, 8,516 rows/s
aggregate vB RMSE initial -> calibrated: 1.9398 -> 1.7324
